# Korean Bill Lifecycle Master Database - Tutorial

This notebook demonstrates how to use the Korean Bill Lifecycle Master Database, which tracks the full lifecycle of bills in the Korean National Assembly (국회). The database covers assemblies 17-22 (2004-present), with the 22nd assembly (22대, 2024-) having the most detailed records.

**Data sources**: 열린국회정보 Open API (8 endpoints combined)

**Contents**:
1. Data Loading and Overview
2. Cross-Assembly Trends (17-22대)
3. 22nd Assembly Deep Dive
4. Committee Meetings Analysis
5. Lifecycle Funnel

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings

try:
    import scienceplots
    plt.style.use(['science', 'no-latex'])
except ImportError:
    plt.style.use('seaborn-v0_8-whitegrid')

# Korean font support
plt.rcParams['font.family'] = 'Nanum Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.figsize'] = (7, 4.5)

# Okabe-Ito colorblind-safe palette
OI = ['#E69F00', '#56B4E9', '#009E73', '#0072B2', '#D55E00', '#CC79A7', '#000000']

warnings.filterwarnings('ignore', category=FutureWarning)

DATA_DIR = 'data/processed'

---
## 1. Data Loading and Overview

The 22nd assembly master table contains 17,205 bills with 54 columns covering identifiers, proposer information, lifecycle timestamps, processing results, vote details, and committee assignments.

In [ ]:
bills22 = pd.read_parquet(f'{DATA_DIR}/master_bills_22.parquet')
print(f"Shape: {bills22.shape[0]:,} bills x {bills22.shape[1]} columns")
print(f"Period: {bills22['ppsl_dt'].min().strftime('%Y-%m-%d')} ~ {bills22['ppsl_dt'].max().strftime('%Y-%m-%d')}")
print(f"\nBill types:")
print(bills22['bill_kind'].value_counts().to_string())
print(f"\nProposer types:")
print(bills22['ppsr_kind'].value_counts().to_string())
print(f"\nCurrent status:")
print(bills22['status'].value_counts().to_string())

### Lifecycle field coverage

The table below shows what fraction of bills have data at each lifecycle stage. Low coverage at later stages reflects the legislative funnel: most bills never make it past the committee stage.

In [ ]:
# Key lifecycle fields and their coverage
lifecycle_cols = {
    'ppsl_dt': '발의일 (Proposal)',
    'committee_dt': '소관위 회부 (Committee referral)',
    'cmt_present_dt': '소관위 상정 (Committee tabling)',
    'cmt_proc_dt': '소관위 처리 (Committee decision)',
    'law_submit_dt': '법사위 회부 (Judiciary referral)',
    'law_proc_dt': '법사위 처리 (Judiciary decision)',
    'rgs_prsnt_dt': '본회의 상정 (Plenary tabling)',
    'proc_dt': '최종 처리 (Final decision)',
    'prom_dt': '공포 (Promulgation)',
}

n = len(bills22)
coverage = pd.DataFrame([
    {'Stage': label, 'Non-null': bills22[col].notna().sum(),
     'Coverage (%)': f"{bills22[col].notna().sum() / n * 100:.1f}%"}
    for col, label in lifecycle_cols.items()
])
coverage

In [ ]:
# Sample rows
bills22[['bill_no', 'bill_nm', 'ppsr_kind', 'rst_proposer', 'ppsl_dt', 'status', 'committee_nm']].head(10)

---
## 2. Cross-Assembly Trends (17-22대)

We combine the lite master tables (17-21대) with the 22nd assembly full master to examine long-run trends. Note that the 22nd assembly is still in session, so its numbers are incomplete.

In [ ]:
# Load lite masters for assemblies 17-21
common_cols = ['bill_id', 'age', 'bill_kind', 'ppsr_kind', 'ppsl_dt',
               'status', 'passed', 'enacted', 'days_to_proc', 'committee_nm']

frames = []
for age in range(17, 22):
    df = pd.read_parquet(f'{DATA_DIR}/master_bills_{age}_lite.parquet')
    frames.append(df[common_cols])

# Add 22nd assembly
frames.append(bills22[common_cols])
all_bills = pd.concat(frames, ignore_index=True)

# Assembly labels and date ranges
assembly_info = {
    17: '17대\n(2004-08)',  18: '18대\n(2008-12)',
    19: '19대\n(2012-16)',  20: '20대\n(2016-20)',
    21: '21대\n(2020-24)',  22: '22대\n(2024-)*',
}
all_bills['label'] = all_bills['age'].map(assembly_info)

print(f"Total bills across all assemblies: {len(all_bills):,}")
print(all_bills.groupby('age').size().to_string())

### Bills per assembly

The number of bills introduced has grown dramatically since the 17th assembly. The 22nd assembly figure is partial since it is still in session.

In [ ]:
agg = all_bills.groupby('age').agg(
    n_bills=('bill_id', 'size'),
    label=('label', 'first')
).reset_index()

fig, ax = plt.subplots(figsize=(6, 3.5))
bars = ax.bar(agg['label'], agg['n_bills'], color=OI[:6], edgecolor='white', linewidth=0.5)

# Annotate bar values
for bar, val in zip(bars, agg['n_bills']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 300,
            f'{val:,}', ha='center', va='bottom', fontsize=8)

ax.set_ylabel('Number of bills')
ax.set_ylim(0, agg['n_bills'].max() * 1.15)
ax.tick_params(axis='x', labelsize=8)

# Mark the 22nd assembly as in-progress
bars[-1].set_hatch('//')
bars[-1].set_alpha(0.7)

plt.tight_layout()
plt.show()

### Passage rate trend

We use two definitions of "passage":
- **Broad (passed)**: includes 원안가결 + 수정가결 + 대안반영폐기 (content reflected in an alternative bill)
- **Narrow (enacted)**: only 원안가결 + 수정가결 (the bill itself becomes law)

The 22nd assembly rates are provisional since most bills are still pending.

In [ ]:
rates = all_bills.groupby('age').agg(
    n=('bill_id', 'size'),
    passed_rate=('passed', 'mean'),
    enacted_rate=('enacted', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(rates['age'], rates['passed_rate'] * 100, 'o-', color=OI[3],
        label='Broad (passed)', markersize=6, linewidth=1.5)
ax.plot(rates['age'], rates['enacted_rate'] * 100, 's--', color=OI[4],
        label='Narrow (enacted)', markersize=6, linewidth=1.5)

# Mark 22대 as in-progress
ax.axvspan(21.7, 22.3, alpha=0.1, color='gray')
ax.text(22, rates.loc[rates['age']==22, 'passed_rate'].values[0]*100 + 3,
        'in session', ha='center', fontsize=7, color='gray')

ax.set_xticks(rates['age'])
ax.set_xticklabels([f"{a}대" for a in rates['age']])
ax.set_ylabel('Rate (%)')
ax.set_xlabel('Assembly')
ax.legend(fontsize=8, loc='upper right')
ax.set_ylim(0, 80)

plt.tight_layout()
plt.show()

### Proposer type composition over time

Korean bills are introduced by four main proposer types: legislators (의원), committee chairs (위원장), the government (정부), and the speaker (의장).

In [ ]:
proposer_types = ['의원', '위원장', '정부', '의장']
prop_counts = (all_bills[all_bills['ppsr_kind'].isin(proposer_types)]
               .groupby(['age', 'ppsr_kind']).size()
               .unstack(fill_value=0))
prop_pct = prop_counts.div(prop_counts.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(6, 3.5))
bottom = np.zeros(len(prop_pct))
colors = {'의원': OI[3], '정부': OI[0], '위원장': OI[2], '의장': OI[5]}

for ptype in ['의원', '정부', '위원장', '의장']:
    vals = prop_pct[ptype].values
    ax.bar([f"{a}대" for a in prop_pct.index], vals, bottom=bottom,
           label=ptype, color=colors[ptype], edgecolor='white', linewidth=0.5)
    bottom += vals

ax.set_ylabel('Share (%)')
ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

---
## 3. 22nd Assembly Deep Dive

### Committee-level analysis

Bills count, passage rate, and average processing time by standing committee. Only committees with at least 100 bills are shown.

In [ ]:
cmt = (bills22[bills22['committee_nm'].notna()]
       .groupby('committee_nm')
       .agg(n_bills=('bill_id', 'size'),
            passage_rate=('passed', 'mean'),
            avg_days=('days_to_proc', 'mean'))
       .reset_index())

# Filter to committees with >= 100 bills
cmt = cmt[cmt['n_bills'] >= 100].sort_values('n_bills', ascending=True)

fig, axes = plt.subplots(1, 3, figsize=(12, 5), sharey=True)

# Panel (a): Bill count
axes[0].barh(cmt['committee_nm'], cmt['n_bills'], color=OI[3], height=0.7)
axes[0].set_xlabel('Bills introduced')
for i, v in enumerate(cmt['n_bills']):
    axes[0].text(v + 15, i, str(v), va='center', fontsize=6.5)

# Panel (b): Passage rate
axes[1].barh(cmt['committee_nm'], cmt['passage_rate'] * 100, color=OI[2], height=0.7)
axes[1].set_xlabel('Passage rate (%)')
axes[1].axvline(bills22['passed'].mean() * 100, color=OI[6], linestyle='--',
                linewidth=0.8, alpha=0.5)

# Panel (c): Average processing days
valid_days = cmt[cmt['avg_days'].notna()]
axes[2].barh(valid_days['committee_nm'], valid_days['avg_days'], color=OI[0], height=0.7)
axes[2].set_xlabel('Avg. days to final decision')

for ax, label in zip(axes, ['(a)', '(b)', '(c)']):
    ax.tick_params(axis='y', labelsize=7.5)
    ax.text(0.02, 0.97, label, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')

plt.tight_layout()
plt.show()

### Passage rate by proposer type

Government bills (정부) typically have much higher passage rates than legislator-sponsored bills (의원). Committee chair bills (위원장) are omnibus alternatives that consolidate multiple legislator bills, so they pass at near-100% rates.

In [ ]:
prop_stats = (bills22[bills22['ppsr_kind'].isin(proposer_types)]
             .groupby('ppsr_kind')
             .agg(n=('bill_id', 'size'),
                  passed_rate=('passed', 'mean'),
                  enacted_rate=('enacted', 'mean'))
             .reindex(['의원', '정부', '위원장', '의장']))

fig, ax = plt.subplots(figsize=(5, 3.5))
x = np.arange(len(prop_stats))
w = 0.35

ax.bar(x - w/2, prop_stats['passed_rate'] * 100, w,
       label='Broad (passed)', color=OI[3], edgecolor='white')
ax.bar(x + w/2, prop_stats['enacted_rate'] * 100, w,
       label='Narrow (enacted)', color=OI[4], edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels([f"{k}\n(n={v:,})" for k, v in
                    zip(prop_stats.index, prop_stats['n'])], fontsize=8)
ax.set_ylabel('Rate (%)')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

### Monthly bill submission trend

How the pace of bill introduction has evolved over the 22nd assembly's term so far.

In [ ]:
monthly = (bills22.set_index('ppsl_dt')
          .resample('ME')
          .size()
          .reset_index(name='n_bills'))
monthly['month_label'] = monthly['ppsl_dt'].dt.strftime('%Y-%m')

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(len(monthly)), monthly['n_bills'], color=OI[3], edgecolor='white', linewidth=0.3)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['month_label'], rotation=45, ha='right', fontsize=6)
ax.set_ylabel('Bills introduced')

# Add mean line
mean_val = monthly['n_bills'].mean()
ax.axhline(mean_val, color=OI[4], linestyle='--', linewidth=0.8, alpha=0.7)
ax.text(len(monthly) - 1, mean_val + 20, f'mean = {mean_val:.0f}',
        ha='right', fontsize=7, color=OI[4])

plt.tight_layout()
plt.show()

### Processing time distribution

Among bills that have reached a final decision, how long does it take from proposal to resolution? We use `days_to_proc` where available, and fall back to the plenary decision date (`rgs_rsln_dt`) for bills without `proc_dt`. Committee-chair bills (위원장) are typically processed within days because they are omnibus alternatives introduced at the point of decision.

In [ ]:
processed = bills22.copy()

# Use days_to_proc where available; otherwise compute from rgs_rsln_dt
processed['days_any'] = processed['days_to_proc']
mask = processed['days_any'].isna() & processed['rgs_rsln_dt'].notna()
processed.loc[mask, 'days_any'] = (
    processed.loc[mask, 'rgs_rsln_dt'] - processed.loc[mask, 'ppsl_dt']
).dt.days
processed = processed[processed['days_any'].notna()]

fig, axes = plt.subplots(1, 2, figsize=(8, 3))

# (a) Overall distribution
axes[0].hist(processed['days_any'], bins=50, color=OI[3], edgecolor='white', linewidth=0.3)
axes[0].axvline(processed['days_any'].median(), color=OI[4], linestyle='--', linewidth=1)
axes[0].text(processed['days_any'].median() + 10, axes[0].get_ylim()[1] * 0.9,
             f"median = {processed['days_any'].median():.0f} days",
             fontsize=7, color=OI[4])
axes[0].set_xlabel('Days from proposal to decision')
axes[0].set_ylabel('Count')

# (b) By proposer type (KDE)
colors_map = {'의원': OI[3], '정부': OI[0], '위원장': OI[2]}
for ptype in ['의원', '정부', '위원장']:
    subset = processed[processed['ppsr_kind'] == ptype]['days_any']
    if len(subset) > 20:
        subset.plot.kde(ax=axes[1], label=f'{ptype} (n={len(subset):,})',
                        color=colors_map[ptype], linewidth=1.2)

axes[1].set_xlabel('Days from proposal to decision')
axes[1].set_ylabel('Density')
axes[1].set_xlim(-50, 700)
axes[1].legend(fontsize=7)

for ax, label in zip(axes, ['(a)', '(b)']):
    ax.text(0.02, 0.97, label, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')

plt.tight_layout()
plt.show()

### Vote margin analysis

Of the 1,236 bills that went to a plenary vote, most pass near-unanimously. Bills with significant opposition (`vote_no > 10`) are relatively rare and politically interesting.

In [ ]:
voted = bills22[bills22['vote_yes'].notna()].copy()
voted['pct_yes'] = voted['vote_yes'] / voted['vote_total'] * 100
voted['contested'] = voted['vote_no'] > 10

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

# (a) Distribution of yes-vote share
axes[0].hist(voted['pct_yes'], bins=40, color=OI[3], edgecolor='white', linewidth=0.3)
axes[0].axvline(50, color='red', linestyle=':', linewidth=0.8, alpha=0.5)
axes[0].set_xlabel('Yes votes (%)')
axes[0].set_ylabel('Count')
axes[0].text(0.98, 0.95, f"n = {len(voted):,} bills with votes",
             transform=axes[0].transAxes, ha='right', va='top', fontsize=7)

# (b) Scatter: yes vs no votes, highlighting contested bills
contested = voted[voted['contested']]
consensus = voted[~voted['contested']]

axes[1].scatter(consensus['vote_yes'], consensus['vote_no'],
                s=8, alpha=0.3, color=OI[1], label=f'Consensus (n={len(consensus)})')
axes[1].scatter(contested['vote_yes'], contested['vote_no'],
                s=12, alpha=0.6, color=OI[4], label=f'Contested (n={len(contested)})')

# Label the most contested bills
top_contested = contested.nlargest(5, 'vote_no')
for _, row in top_contested.iterrows():
    name = row['bill_nm']
    if len(name) > 20:
        name = name[:20] + '...'
    axes[1].annotate(name, (row['vote_yes'], row['vote_no']),
                     fontsize=5, alpha=0.8,
                     xytext=(5, 3), textcoords='offset points')

axes[1].set_xlabel('Yes votes')
axes[1].set_ylabel('No votes')
axes[1].legend(fontsize=7, loc='upper left')

for ax, label in zip(axes, ['(a)', '(b)']):
    ax.text(0.02, 0.97, label, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')

plt.tight_layout()
plt.show()

---
## 4. Committee Meetings Analysis

The `committee_meetings_22` table records each meeting event for bills in committee. A single bill typically goes through multiple meetings (tabling, expert briefing, subcommittee referral, vote, etc.).

In [ ]:
meetings = pd.read_parquet(f'{DATA_DIR}/committee_meetings_22.parquet')
print(f"Committee meetings: {len(meetings):,} rows")
print(f"Unique bills with meeting data: {meetings['bill_id'].nunique():,}")
print(f"\nMeetings per bill:")
mpb = meetings.groupby('bill_id').size()
print(mpb.describe().to_string())

### Meetings per bill distribution and meeting types

Each row in the meetings table represents a single procedural event. The `jrcmit_conf_rslt` column records the type of action taken at that meeting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

# (a) Meetings per bill
mpb = meetings.groupby('bill_id').size()
axes[0].hist(mpb.clip(upper=30), bins=30, color=OI[3], edgecolor='white', linewidth=0.3)
axes[0].set_xlabel('Number of meetings per bill')
axes[0].set_ylabel('Count')
axes[0].text(0.95, 0.95, f'median = {mpb.median():.0f}\nmean = {mpb.mean():.1f}',
             transform=axes[0].transAxes, ha='right', va='top', fontsize=7,
             bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# (b) Meeting type distribution
meeting_types = meetings['jrcmit_conf_rslt'].value_counts()
mt_labels = {
    '상정': 'Tabling (상정)',
    '제안설명': 'Sponsor briefing (제안설명)',
    '검토보고': 'Staff review (검토보고)',
    '소위회부': 'Subcommittee ref. (소위회부)',
    '대체토론': 'General debate (대체토론)',
    '의결': 'Vote/Decision (의결)',
    '축조심사': 'Article-by-article (축조심사)',
    '소위심사보고': 'Subcommittee report (소위심사보고)',
    '찬반토론': 'Pro/con debate (찬반토론)',
}

top_types = meeting_types.head(9)
labels = [mt_labels.get(t, t) for t in top_types.index]
axes[1].barh(labels[::-1], top_types.values[::-1], color=OI[3], height=0.7)
axes[1].set_xlabel('Count')
axes[1].tick_params(axis='y', labelsize=7)

for ax, label in zip(axes, ['(a)', '(b)']):
    ax.text(0.02, 0.97, label, transform=ax.transAxes,
            fontsize=10, fontweight='bold', va='top')

plt.tight_layout()
plt.show()

---
## 5. Lifecycle Funnel

The legislative funnel shows how many of the 17,205 bills in the 22nd assembly have reached each successive stage. This is the defining structural feature of the dataset: most bills die in committee and never reach the plenary floor.

The funnel uses the most complete timestamp available for each stage. Where multiple API sources provide the same timestamp (e.g., `committee_dt` vs `jrcmit_cmmt_dt`), we take the union.

In [ ]:
# Build the funnel using the best available data for each stage
N = len(bills22)

funnel_stages = [
    ('발의\nProposal', N),
    ('소관위 회부\nCommittee referral',
     bills22[['committee_dt', 'jrcmit_cmmt_dt']].notna().any(axis=1).sum()),
    ('소관위 상정\nCommittee tabling',
     bills22[['cmt_present_dt', 'jrcmit_prsnt_dt']].notna().any(axis=1).sum()),
    ('소관위 처리\nCommittee decision',
     bills22[['cmt_proc_dt', 'jrcmit_proc_dt']].notna().any(axis=1).sum()),
    ('법사위 회부\nJudiciary referral',
     bills22[['law_submit_dt', 'law_cmmt_dt']].notna().any(axis=1).sum()),
    ('본회의 상정\nPlenary tabling',
     bills22['rgs_prsnt_dt'].notna().sum()),
    ('최종 가결\nEnacted',
     bills22['enacted'].sum()),
    ('공포\nPromulgated',
     bills22['prom_dt'].notna().sum()),
]

labels, values = zip(*funnel_stages)

fig, ax = plt.subplots(figsize=(7, 5))

# Draw funnel as centered horizontal bars
max_w = values[0]
y_positions = range(len(labels) - 1, -1, -1)

for i, (label, val) in enumerate(zip(labels, values)):
    width = val / max_w
    y = len(labels) - 1 - i
    color = OI[i % len(OI)] if i < len(OI) else OI[0]

    # Centered bar
    ax.barh(y, width, height=0.6, left=(1 - width) / 2,
            color=color, alpha=0.85, edgecolor='white', linewidth=0.5)

    # Label on the right
    ax.text(0.5 + width/2 + 0.02, y,
            f'{val:,} ({val/N*100:.1f}%)',
            va='center', ha='left', fontsize=7.5, fontweight='bold')

ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels[::-1], fontsize=8)
ax.set_xlim(0, 1.3)
ax.set_xticks([])
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_visible(False)

plt.tight_layout()
plt.show()

### Funnel summary table

In [ ]:
funnel_df = pd.DataFrame(funnel_stages, columns=['Stage', 'Count'])
funnel_df['% of Total'] = (funnel_df['Count'] / N * 100).round(1)
funnel_df['Drop-off'] = funnel_df['Count'].diff().fillna(0).astype(int)
funnel_df['Survival from previous (%)'] = (
    funnel_df['Count'] / funnel_df['Count'].shift(1) * 100
).round(1)
funnel_df['Stage'] = funnel_df['Stage'].str.replace('\n', ' - ')
funnel_df

---
## Notes

- **22대 is in session**: The 22nd National Assembly began on 2024-05-30 and is ongoing. Cross-assembly comparisons involving the 22nd should account for the fact that many bills are still pending.
- **`passed` vs `enacted`**: The `passed` variable uses a broad definition that counts 대안반영폐기 (content absorbed into a committee alternative) as passage. The `enacted` variable only counts bills that themselves became law.
- **Satellite tables**: `committee_meetings_22` and `judiciary_meetings_22` provide bill-meeting level data for detailed procedural analysis.
- **Join keys**: `bill_id` is the primary key across all tables. `rst_mona_cd` (legislator code) enables linking to other datasets on Korean legislators.